# 06. Supervised Field Adaptation — ResNet18 (Near-duplicate-safe verified split)

**Research question:** Does fine-tuning with limited labeled PlantDoc data improve the cross-domain performance of ResNet18 models trained on PlantVillage?

**Experiments:**

- **A1 — Baseline → Field:** Adapt the original ResNet18 baseline.
- **A2 — Geometric → Field:** Adapt the ResNet18 model trained with geometric augmentation.

**Data protocol:** The two development partitions come from `prepare_field_dev_grouped.py`, which keeps images with identical RGB content in the **same partition**. Source rows that exactly duplicate the original Field Test, if any, are quarantined. The original 47 Field Test images are not used for training, checkpoint selection, or inference. The new adaptation/dev sample counts may differ from 380/96.

**Important:** Re-run the ResNet18 baseline/photometric/geometric Field Dev evaluation on this *new* split before comparing any development metrics. Metrics from the previous, leaking split are not comparable.

**Interpretation boundary:** Field Dev is a development set; improvements measured here are exploratory, not independently confirmed.


## 1. Runtime and dependencies

Run this notebook using a **Google Colab GPU runtime**. Data exploration, split preparation, and final report assembly can remain on the local machine.

In [ ]:
%pip install -q datasets pandas scikit-learn matplotlib Pillow ImageHash

import sys
import torch
import torchvision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a Colab GPU runtime before training."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Repository synchronization

Run `python src/prepare_field_dev_verified.py --replace-existing` locally first. Commit/push the newly generated `field_adapt_train.csv`, `field_dev.csv`, `field_dev_protocol.json`, and `field_quarantined_train.csv`, along with the grouped splitter script. The notebook verifies the pHash-grouped split and never silently regenerates it.

**Important:** Conflicting-label groups and any training groups close to original Field Test images are quarantined. Original Field Test stays unchanged. Previous Field Dev evaluations/adaptation checkpoints are invalid for this new split.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/thutrang147/plant-disease-diagnosis.git"
PROJECT_ROOT = Path("/content/plant-disease-diagnosis")

if not PROJECT_ROOT.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)

GIT_SHA = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
).strip()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Repository:", PROJECT_ROOT)
print("Git commit:", GIT_SHA)

## 3. Configuration and fixed label mapping

Restore the **previously trained** checkpoints from Google Drive. The source model architectures and label order must match the five-class experiment. Keep outputs from A1 and A2 separate.

In [ ]:
from google.colab import drive
import hashlib
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import InterpolationMode
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from src.dataset import PlantDiseaseDataset

drive.mount("/content/drive")

SEED = 42
BATCH_SIZE = 32
HEAD_EPOCHS = 2
LAST_BLOCK_EPOCHS = 6
EARLY_STOP_PATIENCE = 3
NUM_WORKERS = 0
DEVICE = torch.device("cuda")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
DRIVE_ROOT = Path("/content/drive/MyDrive/plant-disease-diagnosis")
CHECKPOINTS = {
    "A1_baseline": DRIVE_ROOT / "resnet18_baseline" / "resnet18_lab_best.pt",
    "A2_geometric": DRIVE_ROOT / "resnet18_aug_geometric" / "resnet18_best.pt",
}
OUTPUT_ROOT = DRIVE_ROOT / "field_adaptation_verified"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for path in CHECKPOINTS.values():
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint missing on Google Drive: {path}")

label_map = json.loads((SPLITS_DIR / "label_map.json").read_text(encoding="utf-8"))
CLASS_NAMES = [name for name, idx in sorted(label_map.items(), key=lambda item: item[1])]
assert len(CLASS_NAMES) == 5 and sorted(label_map.values()) == list(range(5))
print("Class order:", CLASS_NAMES)
print("Checkpoint files:", *[str(p) for p in CHECKPOINTS.values()], sep="\n - ")

## 4. Dataset partition integrity

The development split is grouped by perceptual pHash (Hamming distance <= 4) across original Field Train and Test, rather than row IDs alone. We check row-ID and pHash-group and exact-content-hash disjointness, label mapping, protocol consistency and complete accounting of the original training rows. Grouped splitting may shift the sample counts from the former 380/96 split. A later image-content audit also recomputes hashes directly from the source dataset.


In [ ]:
required_files = [
    "field_train.csv", "field_adapt_train.csv", "field_dev.csv", "field_test.csv",
    "lab_val.csv", "field_dev_protocol.json", "label_map.json",
    "field_quarantined_train.csv",
]
for filename in required_files:
    if not (SPLITS_DIR / filename).is_file():
        raise FileNotFoundError(
            f"Missing {filename}. Push the grouped split files from your Mac to GitHub."
        )

protocol = json.loads((SPLITS_DIR / "field_dev_protocol.json").read_text(encoding="utf-8"))
if not protocol.get("strategy", "").startswith("Perceptual pHash connected-component grouping"):
    raise ValueError("This notebook requires the near-duplicate-safe split from prepare_field_dev_verified.py")

manifests = {
    name: pd.read_csv(SPLITS_DIR / f"{name}.csv")
    for name in ("field_train", "field_adapt_train", "field_dev", "field_test", "lab_val")
}
quarantine_df = pd.read_csv(SPLITS_DIR / "field_quarantined_train.csv")

assert len(manifests["field_test"]) == 47, "Original Field Test must remain unchanged"
assert len(manifests["field_train"]) == protocol["original_field_train_images"]
assert len(manifests["field_adapt_train"]) == protocol["adaptation_train_images"]
assert len(manifests["field_dev"]) == protocol["development_images"]
assert len(quarantine_df) == protocol["quarantined_total_train_images"]
assert (
    len(manifests["field_adapt_train"]) + len(manifests["field_dev"]) + len(quarantine_df)
    == len(manifests["field_train"])
), "All original training rows must be accounted for"

for name in ("field_adapt_train", "field_dev", "field_test"):
    frame = manifests[name]
    assert (frame["source"] == "plantdoc").all(), name
    assert frame["row_id"].is_unique, name
    assert set(frame["label_id"]) == set(range(5)), name
    assert (frame["class_name"].map(label_map) == frame["label_id"]).all(), name

train = manifests["field_adapt_train"]
dev = manifests["field_dev"]
test = manifests["field_test"]
assert {"exact_hash", "phash", "group_id"}.issubset(train.columns)
assert {"exact_hash", "phash", "group_id"}.issubset(dev.columns)
assert set(train["exact_hash"]).isdisjoint(set(dev["exact_hash"])), "Exact image leakage"
assert set(train["group_id"]).isdisjoint(set(dev["group_id"])), "Near-duplicate leakage"
assert set(train["group_id"]).isdisjoint(set(quarantine_df["group_id"])), "Quarantine leakage"
assert set(dev["group_id"]).isdisjoint(set(quarantine_df["group_id"])), "Quarantine leakage"
assert set(train["row_id"]).isdisjoint(set(dev["row_id"]))
assert set(train["row_id"]).isdisjoint(set(test["row_id"]))
assert set(dev["row_id"]).isdisjoint(set(test["row_id"]))
assert set(train["row_id"]) | set(dev["row_id"]) | set(quarantine_df["row_id"]) == set(manifests["field_train"]["row_id"])

split_hashes = {
    name: hashlib.sha256((SPLITS_DIR / f"{name}.csv").read_bytes()).hexdigest()
    for name in ("field_adapt_train", "field_dev")
}

combined = pd.concat([
    train.assign(partition="adapt_train"),
    dev.assign(partition="dev"),
], ignore_index=True)
print(pd.crosstab(combined["class_name"], combined["partition"]))
print(f"Grouped split verified: adapt={len(train)}, dev={len(dev)}, "
      f"quarantined={len(quarantine_df)}, original test={len(test)} (untouched).")
print("Near-duplicate group overlap between Adapt Train and Dev: 0")


## 5. Image preprocessing and DataLoaders

Use **the same moderate adaptation-time augmentations** for A1 and A2: mild random cropping, horizontal flips, and color jitter. Use the original ImageNet-1k evaluation transform for Field Dev and Lab Validation. This isolates the effect of the *starting checkpoint* during adaptation.

In [ ]:
WEIGHTS = ResNet18_Weights.IMAGENET1K_V1
val_transform = WEIGHTS.transforms()
mean, std = val_transform.mean, val_transform.std

train_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.RandomResizedCrop(
        224, scale=(0.85, 1.0), ratio=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10)
    ], p=0.35),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

adapt_train_ds = PlantDiseaseDataset(
    SPLITS_DIR / "field_adapt_train.csv", transform=train_transform
)
field_dev_ds = PlantDiseaseDataset(
    SPLITS_DIR / "field_dev.csv", transform=val_transform
)
lab_val_ds = PlantDiseaseDataset(
    SPLITS_DIR / "lab_val.csv", transform=val_transform
)

train_loader = DataLoader(
    adapt_train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
field_dev_loader = DataLoader(
    field_dev_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
lab_val_loader = DataLoader(
    lab_val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

images, labels = next(iter(train_loader))
assert images.shape[1:] == (3, 224, 224)
print("Adapt train:", len(adapt_train_ds))
print("Field dev:", len(field_dev_ds))
print("Lab validation:", len(lab_val_ds))
print("Batch shape:", tuple(images.shape))

## 6. Image-content audit (training/development only)

**Purpose:** Verify that RGB pixel SHA-256 fingerprints stored in the grouped manifests match the actual PlantDoc images, with no exact train/dev overlap. Perceptual-hash candidates are exported for manual review; pHash similarity alone does not prove a duplicate. Original Field Test images are not used for model inference, checkpoint selection or hyperparameter tuning.


In [ ]:
import imagehash

source = adapt_train_ds.source_datasets["plantdoc"]

def image_fingerprints(frame):
    records = []
    for row in frame.itertuples(index=False):
        image = source[int(row.row_id)]["image"].convert("RGB")
        content_hash = hashlib.sha256(
            f"{image.size}".encode("utf-8") + image.tobytes()
        ).hexdigest()
        records.append({
            "row_id": int(row.row_id),
            "class_name": str(row.class_name),
            "exact_hash": content_hash,
            "phash": imagehash.phash(image),
        })
    return records

train_hashes = image_fingerprints(manifests["field_adapt_train"])
dev_hashes = image_fingerprints(manifests["field_dev"])
for name, records in [("field_adapt_train", train_hashes), ("field_dev", dev_hashes)]:
    expected = dict(zip(manifests[name]["row_id"], manifests[name]["exact_hash"]))
    for record in records:
        if record["exact_hash"] != expected[record["row_id"]]:
            raise ValueError(f"{name}: stored hash mismatch for row_id={record['row_id']}")

train_exact = {r["exact_hash"] for r in train_hashes}
exact_overlap = [r for r in dev_hashes if r["exact_hash"] in train_exact]
print("Exact cross-partition image duplicates:", len(exact_overlap))
if exact_overlap:
    raise ValueError(
        "Exact duplicate images found across Field Adapt Train and Field Dev. "
        "Fix the dataset split by image group and re-run all Field Dev comparisons."
    )

near_duplicates = [
    {
        "train_row_id": left["row_id"],
        "dev_row_id": right["row_id"],
        "train_class": left["class_name"],
        "dev_class": right["class_name"],
        "phash_distance": left["phash"] - right["phash"],
    }
    for left in train_hashes for right in dev_hashes
    if left["phash"] - right["phash"] <= 4
]
near_df = pd.DataFrame(near_duplicates)
near_path = OUTPUT_ROOT / "possible_near_duplicates.csv"
near_df.to_csv(near_path, index=False)
print("Potential near-duplicate pairs across Train/Dev (expected 0):", len(near_df))
if len(near_df):
    raise ValueError("pHash near-duplicates still cross the Train/Dev boundary! Fix split before training.")
if len(near_df):
    display(near_df.sort_values("phash_distance").head(15))
print("Audit report:", near_path)
# Only image fingerprint checking on original Field Test; no classification inference.
test_hashes = image_fingerprints(manifests["field_test"])
train_dev_hashes = train_hashes + dev_hashes
near_test_pairs = [
    (a["row_id"], b["row_id"], a["phash"] - b["phash"])
    for a in train_dev_hashes for b in test_hashes
    if a["phash"] - b["phash"] <= int(protocol["phash_threshold"])
]
if near_test_pairs:
    raise ValueError(f"Leakage against original Field Test: {near_test_pairs[:10]}")
print("Near-duplicate pairs against original Field Test: 0")


## 7. Model initialization and evaluation helpers

Each experiment starts from its own **frozen source checkpoint**, not from the other adapted experiment. For the small PlantDoc training set, BatchNorm running statistics remain frozen. A staged protocol first adapts the classifier head and then fine-tunes `layer4` plus the classifier.

In [ ]:
def load_source_model(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint.get("architecture") != "resnet18":
        raise ValueError(f"Unexpected architecture: {path}")
    if checkpoint.get("class_names") != CLASS_NAMES:
        raise ValueError(f"Label mapping mismatch: {path}")
    if checkpoint.get("pretrained_weights") != "IMAGENET1K_V1":
        raise ValueError(f"Unexpected source preprocessing: {path}")
    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model.to(DEVICE), checkpoint


def set_trainable(model, stage):
    for name, parameter in model.named_parameters():
        if stage == "head":
            parameter.requires_grad = name.startswith("fc.")
        elif stage == "last_block":
            parameter.requires_grad = name.startswith("layer4.") or name.startswith("fc.")
        else:
            raise ValueError(stage)


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    if training:
        # Frozen BN statistics reduce instability with a small field dataset.
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, truths, predictions = 0.0, [], []

    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * len(targets)
        truths.extend(targets.detach().cpu().tolist())
        predictions.extend(logits.detach().argmax(dim=1).cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": float(accuracy_score(truths, predictions)),
        "macro_f1": float(f1_score(
            truths, predictions, labels=list(range(len(CLASS_NAMES))),
            average="macro", zero_division=0,
        )),
    }


def infer(model, loader):
    model.eval()
    truths, predictions, confidence = [], [], []
    with torch.inference_mode():
        for images, targets in loader:
            logits = model(images.to(DEVICE, non_blocking=True))
            prob = torch.softmax(logits, dim=1)
            score, pred = prob.max(dim=1)
            truths.extend(targets.tolist())
            predictions.extend(pred.cpu().tolist())
            confidence.extend(score.cpu().tolist())
    return np.asarray(truths), np.asarray(predictions), np.asarray(confidence)

## 8. Controlled adaptation experiments

The only difference between A1 and A2 is the **initial PlantVillage checkpoint**. Both use identical training data, augmentation, optimizer settings, seed, and validation protocol.

To avoid losing a strong source checkpoint, the unadapted model is retained as the initial candidate. Checkpoint selection uses **Field Dev macro-F1** only. Early stopping is applied during the last-block stage. Lab Validation is measured after selection to monitor potential forgetting, not to select the adapted checkpoint.

In [ ]:
def save_best(model, target_path, name, source_path, phase, epoch, field_f1):
    torch.save({
        "model_state_dict": model.state_dict(),
        "class_names": CLASS_NAMES,
        "architecture": "resnet18",
        "pretrained_weights": "IMAGENET1K_V1",
        "source_checkpoint": str(source_path),
        "adaptation_experiment": name,
        "best_field_dev_macro_f1": float(field_f1),
        "selected_phase": phase,
        "selected_epoch": int(epoch),
        "seed": SEED,
        "git_commit": GIT_SHA,
        "field_adapt_train_sha256": split_hashes["field_adapt_train"],
        "field_dev_sha256": split_hashes["field_dev"],
    }, target_path)


def adapt_experiment(name, source_path):
    # Restart random generators to make both schedules comparable.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    experiment_dir = OUTPUT_ROOT / name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_file = experiment_dir / "resnet18_field_best.pt"
    model, original_checkpoint = load_source_model(source_path)
    before_field = run_epoch(model, field_dev_loader)
    before_lab = run_epoch(model, lab_val_loader)

    best_f1 = before_field["macro_f1"]
    selected_phase, selected_epoch = "source_unadapted", 0
    save_best(model, checkpoint_file, name, source_path, selected_phase, selected_epoch, best_f1)
    history = [{
        "phase": "source_unadapted", "epoch": 0,
        "train_loss": np.nan, "train_macro_f1": np.nan,
        "dev_loss": before_field["loss"],
        "dev_accuracy": before_field["accuracy"],
        "dev_macro_f1": before_field["macro_f1"],
    }]

    for stage, n_epochs in (("head", HEAD_EPOCHS), ("last_block", LAST_BLOCK_EPOCHS)):
        # Start the second phase from the best checkpoint of the first phase.
        if stage == "last_block":
            state = torch.load(checkpoint_file, map_location="cpu", weights_only=True)
            model.load_state_dict(state["model_state_dict"], strict=True)
        set_trainable(model, stage)
        if stage == "head":
            optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-4, weight_decay=1e-4)
        else:
            optimizer = torch.optim.AdamW([
                {"params": model.layer4.parameters(), "lr": 1e-5},
                {"params": model.fc.parameters(), "lr": 1e-4},
            ], weight_decay=1e-4)

        no_improvement = 0
        for epoch in range(1, n_epochs + 1):
            train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
            dev_metrics = run_epoch(model, field_dev_loader)
            history.append({
                "phase": stage, "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_macro_f1": train_metrics["macro_f1"],
                "dev_loss": dev_metrics["loss"],
                "dev_accuracy": dev_metrics["accuracy"],
                "dev_macro_f1": dev_metrics["macro_f1"],
            })
            pd.DataFrame(history).to_csv(experiment_dir / "training_history.csv", index=False)
            if dev_metrics["macro_f1"] > best_f1 + 1e-6:
                best_f1 = dev_metrics["macro_f1"]
                selected_phase, selected_epoch = stage, epoch
                save_best(model, checkpoint_file, name, source_path, stage, epoch, best_f1)
                no_improvement = 0
            else:
                no_improvement += 1

            print(f"{name} | {stage} {epoch}/{n_epochs} | "
                  f"train F1={train_metrics['macro_f1']:.4f} | "
                  f"dev F1={dev_metrics['macro_f1']:.4f} | best={best_f1:.4f}")
            if stage == "last_block" and no_improvement >= EARLY_STOP_PATIENCE:
                print("Early stopping: no new best Field Dev macro-F1.")
                break

    best_checkpoint = torch.load(checkpoint_file, map_location="cpu", weights_only=True)
    model.load_state_dict(best_checkpoint["model_state_dict"], strict=True)
    after_field = run_epoch(model, field_dev_loader)
    after_lab = run_epoch(model, lab_val_loader)
    y_true, y_pred, scores = infer(model, field_dev_loader)
    assert np.isclose(after_field["macro_f1"], best_f1, atol=1e-9)

    report = classification_report(
        y_true, y_pred, labels=list(range(5)),
        target_names=CLASS_NAMES, digits=4, zero_division=0,
    )
    (experiment_dir / "field_dev_classification_report.txt").write_text(report, encoding="utf-8")
    predictions_df = manifests["field_dev"][
        ["source", "row_id", "class_name", "label_id"]
    ].copy()
    predictions_df["predicted_label_id"] = y_pred
    predictions_df["confidence"] = scores
    predictions_df["correct"] = y_pred == y_true
    predictions_df.to_csv(experiment_dir / "field_dev_predictions.csv", index=False)

    matrix = confusion_matrix(y_true, y_pred, labels=list(range(5)))
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30,
    )
    ax.set_title(f"ResNet18 {name} — Field Dev")
    fig.tight_layout()
    fig.savefig(experiment_dir / "field_dev_confusion_matrix.png", dpi=160)
    plt.close(fig)

    print(f"\n{name}: before Field Dev F1={before_field['macro_f1']:.4f}, "
          f"selected F1={after_field['macro_f1']:.4f}")
    print(f"Lab Val F1: {before_lab['macro_f1']:.4f} → {after_lab['macro_f1']:.4f}")
    print(f"Selected checkpoint: {selected_phase}, epoch {selected_epoch}")

    return {
        "experiment": name,
        "before_field_dev_macro_f1": before_field["macro_f1"],
        "after_field_dev_macro_f1": after_field["macro_f1"],
        "delta_field_dev_macro_f1": after_field["macro_f1"] - before_field["macro_f1"],
        "before_lab_val_macro_f1": before_lab["macro_f1"],
        "after_lab_val_macro_f1": after_lab["macro_f1"],
        "selected_phase": selected_phase,
        "selected_epoch": selected_epoch,
        "checkpoint": str(checkpoint_file),
    }

## 9. Run A1 and A2

Run both experiments without modifying the development partition, training schedule, or hyperparameters between them. Results are exploratory and are subject to selection optimism because Field Dev is used to select checkpoints.

In [ ]:
experiment_results = []
for name, source_path in CHECKPOINTS.items():
    experiment_results.append(adapt_experiment(name, source_path))

summary_df = pd.DataFrame(experiment_results)
summary_df.to_csv(OUTPUT_ROOT / "adaptation_summary.csv", index=False)
display(summary_df.drop(columns="checkpoint"))
print("Summary:", OUTPUT_ROOT / "adaptation_summary.csv")

## 10. Visualize improvement and forgetting

Compare **before vs. selected checkpoint** on Field Dev and Lab Validation. An adapted model can improve field performance while losing accuracy in the source domain (catastrophic forgetting).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ["A1 Baseline", "A2 Geometric"]
x = np.arange(len(labels))
width = 0.34

axes[0].bar(x - width/2, summary_df["before_field_dev_macro_f1"], width, label="Before")
axes[0].bar(x + width/2, summary_df["after_field_dev_macro_f1"], width, label="Selected")
axes[0].set_title("Field Dev Macro-F1")
axes[0].set_ylim(0, 1)
axes[0].set_xticks(x, labels)
axes[0].legend()

axes[1].bar(x - width/2, summary_df["before_lab_val_macro_f1"], width, label="Before")
axes[1].bar(x + width/2, summary_df["after_lab_val_macro_f1"], width, label="Selected")
axes[1].set_title("Lab Validation Macro-F1")
axes[1].set_ylim(0, 1)
axes[1].set_xticks(x, labels)
axes[1].legend()

fig.suptitle("Supervised Field Adaptation — ResNet18")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "adaptation_comparison.png", dpi=160)
plt.show()

## 11. Study notes and interpretation

After running the notebook, document:

1. Whether each adapted model improved Field Dev macro-F1 over its unadapted checkpoint on the **same verified split**.
2. The observed improvements for each class and the number of correct/incorrect images.
3. Changes in PlantVillage Lab Validation performance after adaptation (possible source-domain forgetting).
4. The exact hash-audit result and any pHash near-duplicate candidates still requiring manual inspection.
5. Limitations: small development set, use of Field Dev for early stopping/model selection, and the need for independent field confirmation.

**Do not compare post-adaptation Field Dev results against the old, overlapping Field Dev metrics.** Re-run unadapted checkpoints using `evaluate_field_dev_grouped.py` first. The original 47-image Field Test is not a fresh confirmation set because it was inspected previously. Follow this up with analogous ViT-Small experiments and a new independent field test set.
